In [1]:
import pandas as pd
from pathlib import Path
import os


MASTER_CSV = "rsidis_bigtable_pass0.csv"        # Input master table
OUTPUT_ROOT = Path("output_root")     # Root directory for generated files


PASS_MAP = {
    8.5831: "4pass",
    10.6716: "5pass",
}

# Columns and order for output CSVs
OUTPUT_COLUMNS = [
    "run",
    "run_type",
    "filename",
    "BCM2_Q",
    "ps5",
    "ps6",
    "h_esing_Eff",
    "p_esing_Eff",
    "hDead",
    "pDead"
]

# Filename template for ROOT files
FILENAME_TEMPLATE = (
    "/Users/juliogutierrez/OneDrive - University of Tennessee/R-sidis_analysis/pass0/ROOTfiles/"
    "skimmed_coin_replay_production_{run}_-1.root"
)


def pass_label_from_ebeam(ebeam):
    """Convert ebeam value to folder label."""
    ebeam = float(ebeam)
    if ebeam in PASS_MAP:
        return PASS_MAP[ebeam]
    if 8.0 <= ebeam < 9.0:
        return "4pass"
    if 10.0 <= ebeam < 11.0:
        return "5pass"
    return f"{ebeam:.2f}GeV"

def z_label(z):
    return f"z{float(z):.3f}".rstrip("0").rstrip(".")

def th_label(theta):
    return f"th{float(theta):.3f}".rstrip("0").rstrip(".")

def build_filename(run, pass_label):
    return FILENAME_TEMPLATE.format(
        pass_label=pass_label, run=int(run)
    )

def get_polarity(run_type):
    """Infer polarity from run_type."""
    run_type = str(run_type).strip().upper()
    if "PI+" in run_type:
        return "pi+"
    elif "PI-" in run_type:
        return "pi-"
    else:
        return "unknown"

# ========== MAIN SCRIPT ==========

def main():
    df = pd.read_csv(MASTER_CSV)

    required_cols = ["ebeam", "z", "thpq", "target", "run", "run_type"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # Infer polarity from run_type
    df["polarity"] = df["run_type"].apply(get_polarity)

    # Group by settings
    groups = df.groupby(["ebeam", "z", "thpq", "polarity", "target"], sort=True)

    for (ebeam, z, theta, polarity, target), group in groups:
        # Create folder structure
        pass_label = pass_label_from_ebeam(ebeam)
        zdir = z_label(z)
        thdir = th_label(theta)
        poldir = str(polarity).replace(" ", "")

        folder = OUTPUT_ROOT / pass_label / zdir / thdir / poldir
        folder.mkdir(parents=True, exist_ok=True)

        # Prepare table
        out_df = group.copy()
        out_df["run"] = out_df["run"].astype(int)
        out_df["filename"] = out_df["run"].apply(lambda r: build_filename(r, pass_label))

        out_df["hDead"] = 1.0
        out_df["pDead"] = 1.0

        # Ensure all expected columns exist
        for c in OUTPUT_COLUMNS:
            if c not in out_df.columns:
                out_df[c] = ""

        out_df = out_df[OUTPUT_COLUMNS]

        # Save file
        outfile = folder / f"{target}.csv"
        out_df.to_csv(outfile, index=False, float_format="%.6g")

        print(f"✅ Saved {outfile} ({len(out_df)} rows)")

    print("\nAll CSVs generated successfully!")

# ========== RUN ==========

if __name__ == "__main__":
    main()


✅ Saved output_root/2.23GeV/z-999/th-999/unknown/Dummy.csv (2 rows)
✅ Saved output_root/2.23GeV/z-999/th-999/unknown/LH2.csv (44 rows)
✅ Saved output_root/4pass/z-999/th-999/unknown/C.csv (2 rows)
✅ Saved output_root/4pass/z-999/th-999/unknown/Dummy.csv (4 rows)
✅ Saved output_root/4pass/z-999/th-999/unknown/HOLE.csv (4 rows)
✅ Saved output_root/4pass/z-999/th-999/unknown/LH2.csv (29 rows)
✅ Saved output_root/4pass/z0.36/th2/pi+/C.csv (22 rows)
✅ Saved output_root/4pass/z0.36/th2/pi+/Cu.csv (24 rows)
✅ Saved output_root/4pass/z0.36/th2/pi+/Dummy.csv (9 rows)
✅ Saved output_root/4pass/z0.36/th2/pi+/LD2.csv (19 rows)
✅ Saved output_root/4pass/z0.36/th2/pi+/LH2.csv (20 rows)
✅ Saved output_root/4pass/z0.36/th2/pi-/Dummy.csv (11 rows)
✅ Saved output_root/4pass/z0.36/th2/pi-/LD2.csv (9 rows)
✅ Saved output_root/4pass/z0.36/th2/pi-/LH2.csv (30 rows)
✅ Saved output_root/4pass/z0.36/th2/unknown/C.csv (6 rows)
✅ Saved output_root/4pass/z0.36/th2/unknown/Cu.csv (6 rows)
✅ Saved output_root/4pass